In [1]:
# conda env spec
"""This notebook transforms the CAMP2Ex SPEC bulk properties file and the 
navigation file into a netCDF format. Manual assigment of environmental 
aerosol state is provided as an extra attribute. The aerosol state was 
assigned by analysing aircraft position relative to MISR AOD's, and analyzing 
forward looking P3 video. Take it or leave it.

Generates CAMP2Ex_SPEC*.nc
"""

import xarray as xr
import numpy as np
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent
RAW_DATA = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DATA = PROJECT_ROOT / 'data' / 'interim'

In [2]:
def load_p3(bulk_props_file, nav_file):
    bulk_props = pd.read_csv(bulk_props_file, skiprows=37, header=0, index_col=0)
    nav = pd.read_csv(nav_file, skiprows=73, header=0, index_col=0)
    bulk_props.index = pd.to_datetime(bulk_props.index, unit='s', origin=pd.Timestamp('2019-10-01'))
    bulk_props.index.name = 'time'
    bulk_props = bulk_props.to_xarray()
    nav.index = pd.to_datetime(nav.index, unit='s', origin=pd.Timestamp('2019-10-01'))
    nav.index.name = 'time'
    nav = nav.to_xarray()
    p3 = xr.merge([bulk_props, nav])
    nan_reff = p3.Reff_um.copy(deep=True).data
    nan_reff[nan_reff == 0] = np.nan
    p3['Reff_um'] = (['time'], nan_reff)
    p3['ext_km'] = (3 * 1e-3 * p3.LWC_gm3) / (2 * 997 * 1e-9 * p3.Reff_um)
    return p3

In [3]:
bulk_props_file = RAW_DATA / 'CAMP2Ex_SPEC' / 'CAMP2Ex-CloudBulkProps_P3B_20191001_R0.ICT'
nav_file = RAW_DATA / 'CAMP2Ex_SPEC' / 'CAMP2EX-MetNav_P3B_20191001_R0.ict'
out_file = INTERIM_DATA / 'CAMP2Ex_SPEC_RF17_preprocessed.nc'

In [4]:
# Load P3 data
p3 = load_p3(bulk_props_file, nav_file)
aerosols = np.zeros(p3.time.data.shape)
aerosols[(p3.time.data < np.datetime64('2019-10-02T00:02:00'))] = np.nan
aerosols[(p3.time.data >= np.datetime64('2019-10-02T00:02:00')) & (p3.time.data < np.datetime64('2019-10-02T00:07:00'))] = 1
aerosols[(p3.time.data >= np.datetime64('2019-10-02T00:07:00')) & (p3.time.data < np.datetime64('2019-10-02T00:13:00'))] = 2
aerosols[(p3.time.data >= np.datetime64('2019-10-02T00:12:00')) & (p3.time.data < np.datetime64('2019-10-02T00:16:00'))] = 1
aerosols[(p3.time.data >= np.datetime64('2019-10-02T00:16:00')) & (p3.time.data < np.datetime64('2019-10-02T00:56:00'))] = 0
aerosols[(p3.time.data >= np.datetime64('2019-10-02T00:56:00')) & (p3.time.data < np.datetime64('2019-10-02T01:02:00'))] = 1
aerosols[(p3.time.data >= np.datetime64('2019-10-02T01:02:00')) & (p3.time.data < np.datetime64('2019-10-02T01:28:00'))] = 2
aerosols[(p3.time.data >= np.datetime64('2019-10-02T01:28:00')) & (p3.time.data < np.datetime64('2019-10-02T01:33:00'))] = 1
aerosols[(p3.time.data >= np.datetime64('2019-10-02T01:33:00')) & (p3.time.data < np.datetime64('2019-10-02T02:00:00'))] = 2
aerosols[(p3.time.data >= np.datetime64('2019-10-02T02:00:00'))] = np.nan
p3['aerosols'] = (['time'], aerosols)
p3.to_netcdf(out_file)
